In [70]:
import pandas
import numpy
import os
import math
import urllib
from datetime import datetime

BOLD = '\033[1m'
END = '\033[0m'

#Marchenko rodion Data Analisys prakt №2

print(BOLD+"MARCHENKO RODION FB-23 PRAKT №2 \"Dataset downloading and basic aggregate functions\"")
print("="*83+END)



#This function loads dataset CSV table from the internet
def LoadClimateDatasetAsCSV(WorkDir,RegionID,BeginYear,EndYear):
    url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?provinceID="+str(RegionID)+"&country=UKR&yearlyTag=Weekly&type=Mean&TagCropland=crop&year1="+str(BeginYear)+"&year2="+str(EndYear)
    CurrentTime = datetime.now().strftime("%d-%m-%y-%H-%M-%S")
    Filename = WorkDir+"/Vegetation-health-index-UKR-provinceNo"+str(RegionID)+"--"+CurrentTime+".csv"
    urllib.request.urlretrieve(url,Filename)
    print("\t» Downloaded: Vegetation-health-index-UKR-provinceNo"+str(RegionID)+"--"+CurrentTime+".csv")
    return Filename


#Tris function reads datasets as pandas frames and performs some preprocessing
def ReadDatasetToCSV(WorkDir):
    DataFrames = []
    headers = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'empty'] #CSV table headers
    print("\n»» "+BOLD+"Loading data from files:"+END)
    for path in os.listdir(WorkDir):
        FullPath = os.path.join(WorkDir, path)
        if (os.path.isfile(FullPath) and path[-4:] == ".csv"): # check if current path is a CSV file
            print(path)
            df = pandas.read_csv(FullPath, header = 1, names = headers)
            df = df.drop(df.loc[df["VHI"] == -1].index) #Drop empty value rows for VHI
            df = df.drop(labels="empty", axis=1) #Drop empty column
            df.loc[0, "Year"] = df["Year"][1] #Fix year value in first row (get rid of html tags)
            df = df.dropna(subset=["VHI"]) #Drop rows with NaN in for VHI
            df = df.astype({"Year": "int32"}) #Convert year column to integer for easy indexation
            
            if (path[39:40] == "-"): #Extract province code from predetermined file name and add it as a column
                df["province"] = int(path[38:39])
            else:
                df["province"] = int(path[38:40])
            
            DataFrames.append(df)
    FullData = pandas.concat(DataFrames) #Coalesce all data into a single dataframe
    FullData = FullData.reset_index(drop=True) #Reset global index without preserving the old one
    print("\n"+"="*75)
    print(FullData)
    return FullData


#This function replaces province IDs inside dataframe for a more readable Name Identifier
def ProvinceIdToNames(FullData):
    ReplacementsDict = {1:"Вінницька", 2:"Волинська", 3:"Дніпропетровська", 4:"Донецька", 5:"Житомирська", 6:"Закарпатська", 7:"Запорізька", 8:"Івано-Франківська", 9:"Київська", 10:"Кіровоградська", 11:"Луганська", 12:"Львівська", 13:"Миколаївська", 14:"Одеська", 15:"Полтавська", 16:"Рівенська", 17:"Сумська", 18:"Тернопільська", 19:"Харківська", 20:"Херсонська", 21:"Хмельницька", 22:"Черкаська", 23:"Чернівецька", 24:"Чернігівська", 25:"Республіка Крим", 26:"м. Київ", 27:"м. Севастополь"}
    FullData["province"].replace(ReplacementsDict, inplace = True)
    print("\n"+"="*75)
    print(FullData)


#This function finds anual extremes of VHI index for a given province and year
def AnualExtremeVHI(FullData,ProvinceName,Year):
    yearset = FullData[(FullData["province"] == ProvinceName) & (FullData["Year"] == Year)]["VHI"]
    MinValue = yearset.min()
    MinValueIndex = yearset[yearset == MinValue].index[0]
    MinValueWeek = int(FullData.iloc[MinValueIndex]["Week"])
    MaxValue = yearset.max()
    MaxValueIndex = yearset[yearset == MaxValue].index[0]
    MaxValueWeek = int(FullData.iloc[MaxValueIndex]["Week"])
    
    print("\n")
    print(yearset.head(10))
    print("="*83)
    print(BOLD+"\tThe minimal VHI index =",MinValue,"on week",MinValueWeek,END+" at index",MinValueIndex,"in dataset.")
    print(BOLD+"\tThe maximal VHI index =",MaxValue,"on week",MaxValueWeek,END+" at index",MaxValueIndex,"in dataset.")
    return [MinValue,MaxValue,MinValueWeek,MaxValueWeek,MinValueIndex,MaxValueIndex]


#This function finds extreme or mild years of low VHI index in all dataset for a given percentage of affected territory
def AlltimeVHIbyProvincePercent(FullData, BeginYear, EndYear, AreaPercent, Severity):
    if Severity == "severe":
        MAXVHI = 15
    elif Severity == "mild":
        MAXVHI = 35
    DroughtsSet = FullData[(FullData["VHI"] <= MAXVHI) & (FullData["VHI"] >= 0) & (FullData["Year"] >= BeginYear) & (FullData["Year"] <= EndYear)]
    DroughtsByYear = DroughtsSet.groupby(["Year"])
    DroughtProvincesByYear = DroughtsByYear["province"].nunique() #Get number of provinces affected by drought
    NumberOfProvinces = round((AreaPercent*27)/100) #Calculate the province count based on total area percent
    
    print("\n")
    print("The annual number of provinces affected by drought is:")
    print(DroughtProvincesByYear)
    print("="*83)
    print("The "+Severity+"ly dry years for "+BOLD+str(AreaPercent)+"%"+END+" of land area are:")
    years = list(DroughtProvincesByYear[DroughtProvincesByYear >= NumberOfProvinces].index) #Get only years with more than certain amount of provinces
    print(BOLD,years,END)
    return years
    

    

#"MAIN" FUNCTION:
#Create temporary directory if it does not exist
CurrentDirectory = os.getcwd()
if (os.getcwd()[-13:] != "/DataLab2-tmp"):
    try:
        os.chdir(CurrentDirectory+"/DataLab2-tmp")
    except:
        try:
            os.makedirs(CurrentDirectory+"/DataLab2-tmp")
            print("»» Created tmp directory /DataLab2-tmp.")
        except FileExistsError:
            # directory already exists
            pass
        os.chdir(CurrentDirectory+"/DataLab2-tmp")
 
print("»» The current working directory is: "+os.getcwd()+".")
print("\n")


#Loading vegetation data for all provinces of Ukraine
while True:
    reload = input("Reload dataset files: Y,N")
    if (reload == "Y" or reload == "y" or reload == "N" or reload == "n" or reload == ""):
        break
        
if (reload == "Y" or reload == "y"):
    for province in range(1,28):
        try:
            Filename = LoadClimateDatasetAsCSV(os.getcwd(),province,1982,2024)
        except:
            print("\t» Failed to load for province №",province)
    print("»» Loaded all!!!")

FullDataSet = ReadDatasetToCSV(os.getcwd())
ProvinceIdToNames(FullDataSet)




MARCHENKO RODION FB-23 PRAKT №2 "Dataset downloading and basic aggregate functions"
»» The current working directory is: /home/rodion/DataLab2-tmp.




Reload dataset files: Y,N n



»» Loading data from files:
Vegetation-health-index-UKR-provinceNo25--10-03-24-02-55-37.csv
Vegetation-health-index-UKR-provinceNo5--10-03-24-02-55-13.csv
Vegetation-health-index-UKR-provinceNo12--10-03-24-02-55-21.csv
Vegetation-health-index-UKR-provinceNo24--10-03-24-02-55-36.csv
Vegetation-health-index-UKR-provinceNo2--10-03-24-02-55-04.csv
Vegetation-health-index-UKR-provinceNo11--10-03-24-02-55-20.csv
Vegetation-health-index-UKR-provinceNo20--10-03-24-02-55-31.csv
Vegetation-health-index-UKR-provinceNo16--10-03-24-02-55-26.csv
Vegetation-health-index-UKR-provinceNo21--10-03-24-02-55-32.csv
Vegetation-health-index-UKR-provinceNo4--10-03-24-02-55-12.csv
Vegetation-health-index-UKR-provinceNo6--10-03-24-02-55-14.csv
Vegetation-health-index-UKR-provinceNo3--10-03-24-02-55-08.csv
Vegetation-health-index-UKR-provinceNo8--10-03-24-02-55-16.csv
Vegetation-health-index-UKR-provinceNo19--10-03-24-02-55-30.csv
Vegetation-health-index-UKR-provinceNo26--10-03-24-02-55-38.csv
Vegetation-health

/tmp/ipykernel_97699/2078073845.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  FullData["province"].replace(ReplacementsDict, inplace = True)


In [71]:
extreme1 = AnualExtremeVHI(FullDataSet,"Кіровоградська",2023)
extreme2 = AnualExtremeVHI(FullDataSet,"Луганська",2023)
years1 = AlltimeVHIbyProvincePercent(FullDataSet,1992,2008,20,"severe")
years2 = AlltimeVHIbyProvincePercent(FullDataSet,1992,2008,75,"mild")



57800    42.97
57801    47.14
57802    50.05
57803    50.70
57804    49.43
57805    46.95
57806    45.64
57807    47.34
57808    49.73
57809    50.39
Name: VHI, dtype: float64
	The minimal VHI index = 35.09 on week 42  at index 57841 in dataset.
	The maximal VHI index = 65.7 on week 28  at index 57827 in dataset.


12797    39.95
12798    41.93
12799    45.51
12800    48.10
12801    48.97
12802    48.17
12803    46.58
12804    45.69
12805    46.33
12806    46.62
Name: VHI, dtype: float64
	The minimal VHI index = 30.06 on week 44  at index 12840 in dataset.
	The maximal VHI index = 66.31 on week 30  at index 12826 in dataset.


The annual number of provinces affected by drought is:
Year
1992    2
1993    3
1994    3
1999    1
2000    6
2003    2
2007    5
Name: province, dtype: int64
The severely dry years for 20% of land area are:
 [2000, 2007] 


The annual number of provinces affected by drought is:
Year
1992    25
1993    25
1994    23
1995    26
1996    27
1997    18
1998     8
1